# Análise de Sensibilidade: Da Geometria ao Computador

**Pesquisa Operacional I · Análise de Sensibilidade**  
**Aluno:** Matheus Sousa Marinho &nbsp;·&nbsp; **Matrícula:** 202206132

---

## Estrutura do notebook

| Parte | Problema | Tarefas |
|-------|----------|--------|
| 1 | **Toyco** (Taha §3.6) | 1.1 Executar e validar · 1.2 Decisão gerencial |
| 2 | **NeuralCloud Bruto** | 2.1 Adaptação computacional · 2.2 Melhor investimento |
| 3 | **NeuralCloud Líquido** | 2.3 Fragilidade do ótimo |


## 0. Setup

Instala `amplpy`, baixa o CPLEX e inicializa o objeto `ampl`. Rode **uma vez** por sessão.

In [1]:
!pip install -q amplpy
from amplpy import AMPL, ampl_notebook
import pandas as pd

ampl = ampl_notebook(
    modules=["cplex"],
    license_uuid="default",
)

AMPL Version 20260520 (Linux-6.8.0-1052-azure, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/home/m9t/Documents/ufg/o-research/09/.venv/lib/python3.12/site-packages/ampl_module_base/bin/ampl.lic".



---
## Parte 1 — Toyco

**Referência:** A Toyco monta trens ($x_1$), caminhões ($x_2$) e carros ($x_3$) em três operações.  
Receitas: \$3, \$2, \$5. Capacidades: 430, 460, 420 min/dia.

### 1.1 Modelo AMPL (`toyco.mod`)

Estrutura indexada: `x[j]` com `j ∈ PROD` e restrições `Capacidade {i ∈ OP}` — o mesmo padrão para o NeuralCloud.

In [2]:
%%writefile toyco.mod
# ---- Toyco: análise de sensibilidade (Taha §3.6) ----
set OP;       # operações  (Op1, Op2, Op3)
set PROD;     # produtos   (Trem, Caminhao, Carro)

param margem {PROD} >= 0;      # receita por unidade ($)
param tempo  {OP, PROD} >= 0;  # min de operação i por unidade do produto j
param cap    {OP} >= 0;        # capacidade diária de cada operação (min/dia)

var x {PROD} >= 0;

maximize z: sum {j in PROD} margem[j] * x[j];

s.t. Capacidade {i in OP}:
    sum {j in PROD} tempo[i,j] * x[j] <= cap[i];

Overwriting toyco.mod


### 1.2 Dados (`toyco.dat`)

In [3]:
%%writefile toyco.dat
set OP   := Op1 Op2 Op3 ;
set PROD := Trem Caminhao Carro ;

param margem :=
    Trem      3
    Caminhao  2
    Carro     5 ;

param tempo : Trem  Caminhao  Carro :=
    Op1        1      2        1
    Op2        3      0        2
    Op3        1      4        0 ;

param cap :=
    Op1  430
    Op2  460
    Op3  420 ;

Overwriting toyco.dat


### 1.3 Resolver com `sens=1`

In [4]:
ampl.reset()
ampl.read("toyco.mod")
ampl.read_data("toyco.dat")

ampl.option["solver"] = "cplex"
ampl.option["cplex_options"] = "sens=1"
ampl.solve()

z = ampl.get_objective("z").value()
print(f"z* = $ {z:,.2f}")

x_star = ampl.get_variable("x").get_values().to_pandas()
x_star.columns = ["x*"]
print("\nAlocação ótima:")
display(x_star)

CPLEX 22.1.2:   alg:sens = 1
CPLEX 22.1.2: optimal solution; objective 1350
3 simplex iterations

suffix up OUT;
suffix down OUT;
suffix current OUT;
suffix sensobj OUT;
suffix senslbhi OUT;
suffix senslblo OUT;
suffix sensubhi OUT;
suffix sensublo OUT;
suffix sensobjhi OUT;
suffix sensobjlo OUT;
suffix sensrhshi OUT;
suffix sensrhslo OUT;
z* = $ 1,350.00

Alocação ótima:


,x*
Caminhao,100
Carro,230
Trem,0


### 1.4 Tabela de Restrições — preço-sombra e faixa de viabilidade

| coluna | atributo AMPL | significado |
|--------|--------------|-------------|
| `b` | `Capacidade.ub` | RHS atual ($b_i$) |
| `y` | `Capacidade.dual` | preço-sombra $y_i$ |
| `rhslo`/`rhshi` | `Capacidade.sensrhslo/hi` | faixa de viabilidade |

In [5]:
df_restr = ampl.get_data(
    "Capacidade.body",
    "Capacidade.ub",
    "Capacidade.dual",
    "Capacidade.sensrhslo",
    "Capacidade.sensrhshi",
).to_pandas()
df_restr.columns = ["body", "b", "y", "rhslo", "rhshi"]
df_restr["folga"]  = df_restr["b"] - df_restr["body"]
df_restr["status"] = df_restr["folga"].apply(
    lambda f: "ativa" if abs(f) < 1e-6 else "folgada"
)
df_restr = df_restr[["b", "folga", "y", "rhslo", "rhshi", "status"]]
df_restr = df_restr.sort_values("y", ascending=False)
display(df_restr.round(4))

,b,folga,y,rhslo,rhshi,status
Op2,460,0,2,440,860,ativa
Op1,430,0,1,230,440,ativa
Op3,420,20,0,400,100000000000000000000,folgada


### 1.5 Tabela de Variáveis — custo reduzido e faixa de otimalidade

In [6]:
df_var = ampl.get_data(
    "x", "x.rc", "x.sensobjlo", "x.sensobjhi"
).to_pandas()
df_var.columns = ["x*", "rc", "objlo", "objhi"]

margem = ampl.get_parameter("margem").to_pandas()
margem.columns = ["c_j"]

df_var = df_var.join(margem)
df_var = df_var[["c_j", "x*", "rc", "objlo", "objhi"]]
display(df_var.round(4))

,c_j,x*,rc,objlo,objhi
Caminhao,2,100,0,0,10
Carro,5,230,0,2.333333,100000000000000000000
Trem,3,0,-4,-100000000000000000000,7


### 1.6 Tarefa 1.2 — Decisão Gerencial

Use **somente** as tabelas acima, sem rodar o modelo de novo.  
Em cada item aplique a **Regra dos Três Elementos**: (1) $y_i$ ou `rc` · (2) variação · (3) faixa.

#### (a) Vale a pena alugar capacidade extra da Op.1 a \$0,80/min?

1. **Preço-sombra:** $y_1 = 1{,}00$ \$/min — cada minuto adicional na Op.1 gera \$1,00 a mais em $z$.
2. **Variação proposta:** alugar $\Delta b_1$ minutos extras a \$0,80/min.
3. **Faixa de validade:** $y_1$ permanece válido enquanto $b_1 \leq 440$ min; como $b_1 = 430$, há margem de até **10 min adicionais**.

**Decisão: sim, vale a pena.** O ganho por minuto alugado (\$1,00) supera o custo (\$0,80), gerando lucro líquido de **\$0,20/min**. Recomenda-se alugar até o limite de 10 min extras (até $b_1 = 440$); acima disso, $y_1$ pode mudar e a análise precisa ser refeita.

#### (b) A gerência sugere subir a Op.2 de 460 para 600 min. Qual o ganho previsto em $z$? E se subir para 900 min?

**Para 600 min:**

1. **Preço-sombra:** $y_2 = 2{,}00$ \$/min.
2. **Variação proposta:** $\Delta b_2 = 600 - 460 = 140$ min.
3. **Faixa de validade:** $y_2$ é válido enquanto $b_2 \leq 860$ min; como $600 \leq 860$, a variação está **dentro da faixa**.

**Ganho previsto:** $\Delta z = y_2 \cdot \Delta b_2 = 2{,}00 \times 140 = \mathbf{\$280}$.

---

**Para 900 min:**

1. **Preço-sombra:** $y_2 = 2{,}00$ \$/min (válido somente até $b_2 = 860$ min).
2. **Variação proposta:** $\Delta b_2 = 900 - 460 = 440$ min (variação pretendida).
3. **Faixa de validade:** margem disponível $= 860 - 460 = 400$ min; como $900 > 860$, a expansão **ultrapassa a faixa**.

**Conclusão:** O preço-sombra $y_2 = 2{,}00$ só garante $\Delta z = 2{,}00 \times 400 = \mathbf{\$800}$ até o limite de 860 min. Entre 860 e 900 min a base ótima muda e $y_2$ assume um novo valor — sem reotimização não é possível prever o ganho nesse intervalo.

#### (c) Por que aumentar a Op.3 não muda $z$, embora tenha folga de apenas 20 min?

1. **Preço-sombra:** $y_3 = 0$ — Op.3 é uma restrição **folgada** (não-ativa): a solução ótima usa apenas 400 dos 420 min disponíveis.
2. **Variação proposta:** $\Delta b_3 > 0$ (qualquer expansão da Op.3).
3. **Faixa de validade:** $y_3 = 0$ é válido para todo $b_3 \geq 400$; a restrição só se tornaria ativa — e $y_3$ passaria a ser positivo — se a capacidade fosse **reduzida** abaixo de 400 min.

**Conclusão:** o que limita a produção são Op.1 ($y_1 = 1$) e Op.2 ($y_2 = 2$), não a Op.3. Como já sobram 20 min na Op.3, ela não é gargalo; adicionar capacidade onde não há gargalo não libera produção extra e, portanto, não altera $z$. O tamanho da folga (20 min) é irrelevante — o que importa é que existe folga, pois isso garante $y_3 = 0$.

---
## Parte 2 — NeuralCloud Bruto

**Contexto:** A NeuralCloud vende planos de GPU (Basic, Pro, Ultra) em três datacenters (DC1, DC2, DC3).  
O modelo **bruto** maximiza a margem bruta semanal, sem descontar custos operacionais.

| | Toyco | NeuralCloud |
|--|--|--|
| Conjuntos | `OP`, `PROD` | `DC`, `PLANO` |
| Variáveis | `var x {PROD}` (1 índice) | `var x {DC, PLANO}` (2 índices) |
| Restrições principais | `Capacidade {OP}` | `Capac {DC}`, `Potencia {DC}`, `Demanda {PLANO}` |
| Restrições auxiliares | (nenhuma) | `BalSup`, `BalInf` (balanço DC1-DC3), `TolFalhas {DC, PLANO}` |

> **Tarefa 2.1:** Adapte o padrão de extração da Toyco para lidar com variáveis duplamente indexadas e múltiplas famílias de restrição.

### 2.1 Modelo (`bruto.mod`)

Seis famílias de restrição:
- **`Capac {DC}`** — limite de instâncias por datacenter
- **`Potencia {DC}`** — limite de potência elétrica (kW) por datacenter
- **`Demanda {PLANO}`** — demanda máxima de mercado por tipo de plano
- **`BalSup` / `BalInf`** — balanço de carga entre DC1 e DC3 (diferença máxima de `tol_balanc` instâncias)
- **`TolFalhas {DC, PLANO}`** — cada DC pode hospedar no máximo `frac_max` da demanda total de cada plano

In [7]:
%%writefile bruto.mod
set DC;
set PLANO;

param margem   {PLANO} >= 0;
param kw       {PLANO} >= 0;
param dem_max  {PLANO} >= 0;
param cap_inst {DC}    >= 0;
param cap_kw   {DC}    >= 0;
param tol_balanc       >= 0;
param frac_max         >= 0, <= 1;

var x {DC, PLANO} >= 0;
var y {i in DC} = sum {j in PLANO} x[i,j];
var carga {j in PLANO} = sum {i in DC} x[i,j];

maximize Margem_Bruta:
    sum {i in DC, j in PLANO} margem[j] * x[i,j];

s.t. Capac    {i in DC}: y[i] <= cap_inst[i];
s.t. Potencia {i in DC}: sum {j in PLANO} kw[j]*x[i,j] <= cap_kw[i];
s.t. Demanda  {j in PLANO}: carga[j] <= dem_max[j];
s.t. BalSup: y["DC1"] - y["DC3"] <= tol_balanc;
s.t. BalInf: y["DC3"] - y["DC1"] <= tol_balanc;
s.t. TolFalhas {i in DC, j in PLANO}: x[i,j] <= frac_max * carga[j];

Overwriting bruto.mod


### 2.2 Dados (`bruto.dat`)

**Margens brutas** ($/instância/semana): Basic=\$80, Pro=\$120, Ultra=\$200  
**Consumo de energia** (kW/instância): Basic=5, Pro=8, Ultra=12  
**Capacidade de instâncias** por DC: DC1=600, DC2=800, DC3=500  
**Capacidade elétrica** (kW) por DC: DC1=4 500, DC2=5 000, DC3=3 000  
**Demanda máxima** por plano: Basic=700, Pro=900, Ultra=600  
**Parâmetros auxiliares:** `tol_balanc=100` instâncias e `frac_max=0,60` (máximo 60% de qualquer plano em um único DC).

In [8]:
%%writefile bruto.dat
set DC    := DC1 DC2 DC3 ;
set PLANO := Basic Pro Ultra ;

param tol_balanc := 100 ;
param frac_max   := 0.60 ;

param :       margem  kw  dem_max :=
  Basic         80     5     700
  Pro          120     8     900
  Ultra        200    12     600 ;

param :  cap_inst  cap_kw :=
  DC1      600      4500
  DC2      800      5000
  DC3      500      3000 ;

Overwriting bruto.dat


### 2.3 Resolver e validar (Tarefa 2.1)

In [9]:
ampl.reset()
ampl.read("bruto.mod")
ampl.read_data("bruto.dat")

ampl.option["solver"] = "cplex"
ampl.option["cplex_options"] = "sens=1"
ampl.solve()

z_bruto = ampl.get_objective("Margem_Bruta").value()
print(f"z* (bruto) = $ {z_bruto:,.2f}")

x_bruto = ampl.get_variable("x").get_values().to_pandas()
x_bruto.columns = ["x*"]
x_bruto = x_bruto[x_bruto["x*"] > 1e-6]
print("\nAlocação ótima (variáveis > 0):")
display(x_bruto)

CPLEX 22.1.2:   alg:sens = 1
CPLEX 22.1.2: optimal solution; objective 203000
8 simplex iterations

suffix up OUT;
suffix down OUT;
suffix current OUT;
suffix sensobj OUT;
suffix senslbhi OUT;
suffix senslblo OUT;
suffix sensubhi OUT;
suffix sensublo OUT;
suffix sensobjhi OUT;
suffix sensobjlo OUT;
suffix sensrhshi OUT;
suffix sensrhslo OUT;
z* (bruto) = $ 203,000.00

Alocação ótima (variáveis > 0):


x*
index0 index1            
DC1    Basic    92.857143
       Pro      90.000000
       Ultra   276.309524
DC2    Basic   420.000000
       Pro     135.000000
       Ultra   151.666667
DC3    Basic   187.142857
       Ultra   172.023810

### 2.4 Tabela de Restrições — todas as famílias

Extraímos cada família separadamente e concatenamos, ordenando por preço-sombra decrescente.  
Esse padrão é idêntico ao da Toyco — apenas o nome da família muda.

In [10]:
def extrai_restr(ampl, familia, sentido="ub"):
    """Extrai tabela de sensibilidade para uma família de restrições."""
    bound = f"{familia}.ub" if sentido == "ub" else f"{familia}.lb"
    df = ampl.get_data(
        f"{familia}.body",
        bound,
        f"{familia}.dual",
        f"{familia}.sensrhslo",
        f"{familia}.sensrhshi",
    ).to_pandas()
    df.columns = ["body", "b", "y", "rhslo", "rhshi"]
    df["folga"]   = (df["b"] - df["body"]).abs()
    df["status"]  = df["folga"].apply(lambda f: "ativa" if f < 1e-6 else "folgada")
    df["família"] = familia
    return df[["família", "b", "folga", "y", "rhslo", "rhshi", "status"]]

df_capac    = extrai_restr(ampl, "Capac")
df_potencia = extrai_restr(ampl, "Potencia")
df_demanda  = extrai_restr(ampl, "Demanda")

df_todas = pd.concat([df_capac, df_potencia, df_demanda])
df_todas = df_todas.sort_values("y", ascending=False)
print("Tabela de restrições (ordenada por y decrescente):")
display(df_todas.round(4))

Tabela de restrições (ordenada por y decrescente):


,família,b,folga,y,rhslo,rhshi,status
Ultra,Demanda,600,0.0000,20.0,347.2222,750.0,ativa
DC3,Potencia,3000,0.0000,15.0,1535.4167,4754.444444,ativa
DC2,Potencia,5000,0.0000,15.0,3200.0000,5861.538462,ativa
DC1,Potencia,4500,0.0000,15.0,2700.0000,5476.388889,ativa
Basic,Demanda,700,0.0000,5.0,357.8947,1060,ativa
DC1,Capac,600,140.8333,0.0,459.1667,100000000000000000000,folgada
DC3,Capac,500,140.8333,0.0,359.1667,100000000000000000000,folgada
DC2,Capac,800,93.3333,0.0,706.6667,100000000000000000000,folgada
Pro,Demanda,900,675.0000,0.0,225.0000,100000000000000000000,folgada


### 2.5 Tabela de Variáveis

O índice agora tem dois níveis `(DC, PLANO)` — o `.to_pandas()` já devolve um MultiIndex.

In [11]:
df_xvar = ampl.get_data(
    "x", "x.rc", "x.sensobjlo", "x.sensobjhi"
).to_pandas()
df_xvar.columns = ["x*", "rc", "objlo", "objhi"]

margem_df = ampl.get_parameter("margem").to_pandas()
margem_df.columns = ["c_j"]
df_xvar["c_j"] = df_xvar.index.get_level_values(-1).map(margem_df["c_j"])

df_xvar = df_xvar[["c_j", "x*", "rc", "objlo", "objhi"]]
df_xvar = df_xvar.sort_values("rc", ascending=True)
print("Tabela de variáveis (ordenada por rc):")
display(df_xvar.round(4))

Tabela de variáveis (ordenada por rc):


c_j        x*   rc                   objlo  \
index0 index1                                               
DC2    Pro     120  135.0000 -0.0                   120.0   
DC1    Ultra   200  276.3095 -0.0                     200   
       Basic    80   92.8571 -0.0                    80.0   
DC2    Basic    80  420.0000 -0.0                      80   
       Ultra   200  151.6667  0.0              166.666667   
DC3    Ultra   200  172.0238  0.0                     200   
       Basic    80  187.1429  0.0                      80   
DC1    Pro     120   90.0000  0.0                     120   
DC3    Pro     120    0.0000  0.0  -100000000000000000000   

                               objhi  
index0 index1                         
DC2    Pro                133.333333  
DC1    Ultra                     200  
       Basic                      80  
DC2    Basic   100000000000000000000  
       Ultra                     200  
DC3    Ultra              293.333333  
       Basic                    80.0  
DC1    Pro                     120.0  
DC3    Pro                     120.0

### 2.6 Tarefa 2.2 — O Melhor Investimento

A NeuralCloud vai ampliar a potência de **um** dos três DCs. Sem rodar o solver novamente, analise a coluna `Potencia[i]` da tabela de restrições:

- **P1. Preço-sombra.** Qual DC ganharia se olhássemos só o maior $y_i$?
- **P2. Faixa.** Quanto cada DC pode crescer antes que o $y_i$ mude ($\Delta b_i^\text{max}$)?
- **P3. Ganho.** Multiplicando $y_i \cdot \Delta b_i^\text{max}$, qual DC oferece o maior retorno financeiro viável no momento?

#### P1 — Preço-sombra: qual DC tem o maior $y_i$?

1. **Preço-sombra:** $y_\text{DC1} = y_\text{DC2} = y_\text{DC3} = 15$ \$/kW — os três DCs estão empatados; cada kW extra de potência gera \$15 adicionais em $z$, independentemente do datacenter.
2. **Variação proposta:** qualquer $\Delta b_i > 0$ em qualquer DC.
3. **Faixa de validade:** o preço-sombra de \$15/kW corresponde à margem por kW do plano Pro ($120/8 = 15$), que é o plano determinante no ótimo. Basic ($80/5 = 16$) e Ultra ($200/12 \approx 16{,}67$) têm razão maior, mas sua alocação é limitada pelas restrições de balanço e tolerância, deixando Pro como pivô da Potência.

**Conclusão parcial:** olhando só $y_i$, não há como distinguir os DCs — todos valem \$15/kW. Para decidir o melhor investimento é preciso combinar o preço-sombra com a faixa de expansão disponível (P2 e P3).

#### P2 — Faixa: quanto cada DC pode crescer ($\Delta b_i^\text{max}$)?

A faixa de validade de $y_i = 15$ vai até `rhshi`. O crescimento máximo dentro dessa faixa é $\Delta b_i^\text{max} = \text{rhshi}_i - b_i$:

| DC | $b_i$ atual (kW) | `rhshi` (kW) | $\Delta b_i^\text{max}$ (kW) |
|----|-----------------|-------------|------------------------------|
| DC1 | 4 500 | 5 476 | **976** |
| DC2 | 5 000 | 5 862 | **862** |
| DC3 | 3 000 | 4 754 | **1 754** |

O DC3 tem a maior margem de expansão segura; o DC2 tem a menor.

#### P3 — Ganho: em qual DC investir?

Multiplicando $y_i \cdot \Delta b_i^\text{max}$:

| DC | $y_i$ (\$/kW) | $\Delta b_i^\text{max}$ (kW) | Ganho máximo garantido |
|----|--------------|------------------------------|------------------------|
| DC1 | 15 | 976 | **\$14 646** |
| DC2 | 15 | 862 | **\$12 923** |
| DC3 | 15 | **1 754** | **\$26 317** |

**Decisão: investir no DC3.** Com o mesmo preço-sombra (\$15/kW), o DC3 oferece a maior janela de expansão segura (1 754 kW), garantindo o maior retorno viável sem reotimização — \$26 317, contra \$14 646 do DC1 e \$12 923 do DC2.

O DC2, apesar de ter maior capacidade instalada atual, possui menos margem de expansão segura (a faixa de validade do seu $y_\text{DC2}$ esgota-se mais cedo). O DC3, menor e com energia mais barata, é o que mais pode crescer dentro do ótimo atual.

> *Confirmação pelo código acima.*

In [12]:
# Auxílio de cálculo para P2 e P3 — execute após preencher a análise acima
df_pot = df_potencia.copy()
df_pot["delta_max"] = df_pot["rhshi"] - df_pot["b"]
df_pot["ganho_max"] = df_pot["y"] * df_pot["delta_max"]
display(df_pot[["b", "y", "rhslo", "rhshi", "delta_max", "ganho_max"]].round(2))
melhor = df_pot["ganho_max"].idxmax()
print(f"\nMelhor investimento: {melhor}  →  ganho máximo = $ {df_pot.loc[melhor, 'ganho_max']:,.2f}")

,b,y,rhslo,rhshi,delta_max,ganho_max
DC1,4500,15,2700.00,5476.39,976.39,14645.83
DC2,5000,15,3200.00,5861.54,861.54,12923.08
DC3,3000,15,1535.42,4754.44,1754.44,26316.67



Melhor investimento: DC3  →  ganho máximo = $ 26,316.67


---
## Parte 3 — NeuralCloud Líquido (Tarefa 2.3)

O modelo **líquido** desconta os custos reais de operação por instância/semana:
- **Energia elétrica** — `preco_kwh[i] * kw[j] * H` (varia por DC e por plano)
- **Água de resfriamento** — `preco_agua[i] * wue[i] * kw[j] * H / 1000` (varia por DC)
- **Depreciação de GPU** — `dep_gpu[j]` por instância/semana (varia por plano)
- **OPEX fixo** — `custo_op[j]` por instância/semana (varia por plano)

A função objetivo passa a ser a **margem líquida semanal** por combinação (DC, Plano).

### 3.1 Modelo (`liquido.mod`)

In [13]:
%%writefile liquido.mod
set DC;
set PLANO;

param preco      {PLANO} >= 0;
param kw         {PLANO} >= 0;
param dem_max    {PLANO} >= 0;
param dep_gpu    {PLANO} >= 0;
param custo_op   {PLANO} >= 0;
param cap_inst   {DC}    >= 0;
param cap_kw     {DC}    >= 0;
param preco_kwh  {DC}    >= 0;
param wue        {DC}    >= 0;
param preco_agua {DC}    >= 0;
param H                  >= 0;
param tol_balanc         >= 0;
param frac_max           >= 0, <= 1;

var x {DC, PLANO} >= 0;
var y {i in DC} = sum {j in PLANO} x[i,j];
var carga {j in PLANO} = sum {i in DC} x[i,j];
var kwh {i in DC} = H * sum {j in PLANO} kw[j]*x[i,j];

maximize Margem_Liquida:
    sum {i in DC, j in PLANO} preco[j] * x[i,j]
  - sum {i in DC} preco_kwh[i] * kwh[i]
  - sum {i in DC} preco_agua[i] * wue[i] * kwh[i] / 1000
  - sum {i in DC, j in PLANO} dep_gpu[j] * x[i,j]
  - sum {i in DC, j in PLANO} custo_op[j] * x[i,j];

s.t. Capac    {i in DC}: y[i] <= cap_inst[i];
s.t. Potencia {i in DC}: sum {j in PLANO} kw[j]*x[i,j] <= cap_kw[i];
s.t. Demanda  {j in PLANO}: carga[j] <= dem_max[j];
s.t. BalSup: y["DC1"] - y["DC3"] <= tol_balanc;
s.t. BalInf: y["DC3"] - y["DC1"] <= tol_balanc;
s.t. TolFalhas {i in DC, j in PLANO}: x[i,j] <= frac_max * carga[j];

Overwriting liquido.mod


### 3.2 Dados (`liquido.dat`)

Modelo semanal com H=168 h. Os custos reais descontados na função objetivo são:

- **Energia** por DC ($/kWh): DC1=0,12 · DC2=0,08 · DC3=0,06
- **WUE** (eficiência hídrica): DC1=1,8 · DC2=1,5 · DC3=0,5
- **Água** por DC ($/m³): DC1=3,00 · DC2=2,50 · DC3=1,80
- **Depreciação GPU** por plano ($/inst/sem): Basic=10 · Pro=30 · Ultra=80
- **OPEX** por plano ($/inst/sem): Basic=5 · Pro=10 · Ultra=20

O DC3 tem energia e água mais baratas; o DC1 tem o maior custo operacional por kW.

In [14]:
%%writefile liquido.dat
set DC    := DC1 DC2 DC3 ;
set PLANO := Basic Pro Ultra ;

param H          := 168 ;
param tol_balanc := 100 ;
param frac_max   := 0.60 ;

param :  preco  kw  dem_max  dep_gpu  custo_op :=
  Basic    90    5    700      10        5
  Pro     180    8    900      30       10
  Ultra   360   12    600      80       20 ;

param :  cap_inst  cap_kw  preco_kwh  wue  preco_agua :=
  DC1      600     4500     0.12     1.8    3.00
  DC2      800     5000     0.08     1.5    2.50
  DC3      500     3000     0.06     0.5    1.80 ;

Overwriting liquido.dat


### 3.3 Margens líquidas por (DC, Plano)

Antes de otimizar, veja a margem líquida de cada combinação para entender o que o solver vai preferir.

In [15]:
# Margem líquida = preco[j] - kw[j]*preco_kwh[i]*H - preco_agua[i]*wue[i]*kw[j]*H/1000 - dep_gpu[j] - custo_op[j]
precos     = {"Basic": 90,  "Pro": 180, "Ultra": 360}
kw_val     = {"Basic":  5,  "Pro":   8, "Ultra":  12}
dep_gpu    = {"Basic": 10,  "Pro":  30, "Ultra":  80}
custo_op   = {"Basic":  5,  "Pro":  10, "Ultra":  20}
kwh_preco  = {"DC1": 0.12, "DC2": 0.08, "DC3": 0.06}
wue_val    = {"DC1": 1.8,  "DC2": 1.5,  "DC3": 0.5}
agua_preco = {"DC1": 3.00, "DC2": 2.50, "DC3": 1.80}
H = 168

rows = []
for dc in ["DC1", "DC2", "DC3"]:
    for pl in ["Basic", "Pro", "Ultra"]:
        c_energia = kw_val[pl] * kwh_preco[dc] * H
        c_agua    = agua_preco[dc] * wue_val[dc] * kw_val[pl] * H / 1000
        margem    = precos[pl] - c_energia - c_agua - dep_gpu[pl] - custo_op[pl]
        rows.append({"DC": dc, "Plano": pl,
                     "preco": precos[pl],
                     "c_energia": round(c_energia, 2),
                     "c_agua":    round(c_agua, 3),
                     "dep+opex":  dep_gpu[pl] + custo_op[pl],
                     "margem_liq": round(margem, 2),
                     "margem/kW":  round(margem / kw_val[pl], 3)})

df_marg = pd.DataFrame(rows).set_index(["DC", "Plano"])
display(df_marg)

preco  c_energia  c_agua  dep+opex  margem_liq  margem/kW
DC  Plano                                                           
DC1 Basic     90     100.80   4.536        15      -30.34     -6.067
    Pro      180     161.28   7.258        40      -28.54     -3.567
    Ultra    360     241.92  10.886       100        7.19      0.599
DC2 Basic     90      67.20   3.150        15        4.65      0.930
    Pro      180     107.52   5.040        40       27.44      3.430
    Ultra    360     161.28   7.560       100       91.16      7.597
DC3 Basic     90      50.40   0.756        15       23.84      4.769
    Pro      180      80.64   1.210        40       58.15      7.269
    Ultra    360     120.96   1.814       100      137.23     11.435

### 3.4 Resolver o modelo líquido

In [16]:
ampl.reset()
ampl.read("liquido.mod")
ampl.read_data("liquido.dat")

ampl.option["solver"] = "cplex"
ampl.option["cplex_options"] = "sens=1"
ampl.solve()

z_liq = ampl.get_objective("Margem_Liquida").value()
print(f"z* (líquido) = $ {z_liq:,.2f}")
print(f"z* (bruto)   = $ {z_bruto:,.2f}")

x_liq = ampl.get_variable("x").get_values().to_pandas()
x_liq.columns = ["x* líquido"]
x_liq = x_liq[x_liq["x* líquido"] > 1e-6]
print("\nAlocação ótima — líquido (variáveis > 0):")
display(x_liq)

CPLEX 22.1.2:   alg:sens = 1
CPLEX 22.1.2: optimal solution; objective 64675.76
7 simplex iterations

suffix up OUT;
suffix down OUT;
suffix current OUT;
suffix sensobj OUT;
suffix senslbhi OUT;
suffix senslblo OUT;
suffix sensubhi OUT;
suffix sensublo OUT;
suffix sensobjhi OUT;
suffix sensobjlo OUT;
suffix sensrhshi OUT;
suffix sensrhslo OUT;
z* (líquido) = $ 64,675.76
z* (bruto)   = $ 203,000.00

Alocação ótima — líquido (variáveis > 0):


x* líquido
index0 index1            
DC1    Pro            150
DC2    Pro            100
       Ultra          350
DC3    Ultra          250

### 3.5 Custo reduzido do plano Basic no modelo líquido

In [17]:
df_rc_liq = ampl.get_data(
    "x", "x.rc", "x.sensobjlo", "x.sensobjhi"
).to_pandas()
df_rc_liq.columns = ["x*", "rc", "objlo", "objhi"]

# Filtrar apenas as linhas do plano Basic
basic_rows = df_rc_liq.xs("Basic", level=-1) if isinstance(df_rc_liq.index, pd.MultiIndex) else \
             df_rc_liq[df_rc_liq.index.get_level_values(-1) == "Basic"]
print("Custo reduzido (rc) do plano Basic por DC:")
display(basic_rows.round(4))
print()
print("Tabela de restrições — modelo líquido:")
df_pot_liq = extrai_restr(ampl, "Potencia")
display(df_pot_liq.round(4))

Custo reduzido (rc) do plano Basic por DC:


,x*,rc,objlo,objhi
index0,,,,
DC1,0,0.000,-32.928,-21.850667
DC2,0,-12.728,-100000000000000000000,17.378
DC3,0,-31.936,-100000000000000000000,55.78



Tabela de restrições — modelo líquido:


,família,b,folga,y,rhslo,rhshi,status
DC1,Potencia,4500,3300,0.0000,1200,100000000000000000000,folgada
DC2,Potencia,5000,0,4.2532,2400,5000,ativa
DC3,Potencia,3000,0,5.3480,2805,3000,ativa


### 3.6 Tarefa 2.3 — A Fragilidade do Ótimo

Compare $z^*$ e a alocação entre os modelos bruto e líquido. Use a tabela de variáveis (coluna `rc`) e de restrições (coluna `y`).

#### (a) O que o custo reduzido do plano Basic diz sobre seu desaparecimento da nova solução?

O solver retorna `rc` e a faixa de otimalidade para o plano Basic em cada DC:

| DC | Margem líquida (\$/inst) | `rc` | `objhi` (limiar p/ entrar) | Interpretação |
|----|--------------------------|------|---------------------------|--------------|
| DC1 | $-30{,}34$ | $0{,}00$ | $-21{,}85$ | degenera — margem $\approx 2\times$ pior que o limiar |
| DC2 | $+4{,}65$ | $-12{,}73$ | $+17{,}38$ | margem positiva, mas perde para Pro/Ultra por kW |
| DC3 | $+23{,}84$ | $-31{,}94$ | $+55{,}78$ | DC3 é disputado por Ultra e Pro, Basic não consegue entrar |

1. **Custo reduzido:** `rc = 0` para DC1 e `rc < 0` para DC2 e DC3. Basic está fora da base ($x^* = 0$) em todos os DCs; o `rc` mede o déficit de margem efetiva que impede a entrada.
2. **DC1 — caso degenerado:** `rc = 0` não significa que Basic seria indiferente — significa que a base ótima está num vértice degenerado. A margem líquida de Basic em DC1 é $-30{,}34$ \$/inst/sem (energia sozinha custa $5 \times 0{,}12 \times 168 = \$100{,}80$, mais água e OPEX). O `objhi = -21{,}85` revela que a margem precisaria melhorar em pelo menos **\$8,49** (de $-30{,}34$ para $-21{,}85$) antes de Basic entrar, mesmo sendo um ponto degenerado.
3. **DC2 e DC3:** margens positivas, mas Basic concorre em desvantagem com Pro e Ultra por potência elétrica — esses planos entregam mais margem líquida por kW consumido (DC3 Ultra: \$11,44/kW; DC3 Pro: \$7,27/kW; DC3 Basic: \$4,77/kW). O `objhi` indica o patamar de margem em que Basic passaria a ser competitivo.

**Conclusão:** Basic desaparece porque, em DC1, a conta de energia já torna a margem negativa; em DC2 e DC3, Basic simplesmente perde na disputa por potência para planos mais rentáveis por kW. O `rc` (ou o limiar em `objhi`) quantifica esse déficit em cada caso.

#### (b) O gargalo operacional mudou de lugar? O que isso alerta um engenheiro que projeta datacenters olhando apenas para receita bruta?

**No modelo bruto**, os três DCs tinham $y_\text{Potencia} = 15$ — potência elétrica era igualmente valiosa em todos os datacenters.

**No modelo líquido**, os preços-sombra divergem drasticamente:

| DC | $y_\text{Potencia}$ | folga (kW) | Status |
|----|--------------------:|----------:|--------|
| DC1 | $0{,}00$ | 3 300 | **folgada** |
| DC2 | $4{,}25$ | 0 | ativa |
| DC3 | $5{,}35$ | 0 | ativa (gargalo) |

O DC1 — antes um dos três gargalos — torna-se **restrição folgada**: a solução ótima aloca apenas 150 instâncias Pro no DC1 (consumindo $150 \times 8 = 1\,200$ kW dos 4 500 disponíveis), deixando 3 300 kW ociosos. O custo de energia do DC1 (\$0,12/kWh) é o dobro do DC3 (\$0,06/kWh), tornando-o pouco atrativo mesmo com grande capacidade.

O gargalo **migra** para DC3 (energia mais barata a \$0,06/kWh, WUE = 0,5), que agora tem o maior preço-sombra ($y = 5{,}35$). Cada kW a mais em DC3 vale $5{,}35$, quase 26% mais do que em DC2 ($4{,}25$) — e infinitamente mais do que em DC1 (que tem folga e $y = 0$).

**Alerta ao engenheiro:** um projeto dimensionado pela receita bruta enxerga os três DCs como igualmente valiosos por kW instalado (preços-sombra empatados em \$15/kW). Na realidade, após incluir energia e resfriamento, o retorno marginal de um kW a mais pode ser zero em DC1 e positivo apenas em DC2/DC3. O engenheiro que usa o modelo bruto tende a superdimensionar DC1 e subdimensionar DC3, perdendo margem operacional em cada ampliação de capacidade.

---
**Referências:** H. Taha, *Pesquisa Operacional*, 8ª Ed. · [amplpy](https://amplpy.readthedocs.io/) · IBM CPLEX 22.1